In [ ]:
# CELL 1: Verify T4 GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU detected. Set Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# CELL 2: Where the training tools' credentials come from (Colab Secrets).
# Left sidebar -> key icon -> add these and turn ON "Notebook access":
#   GH_TOKEN         GitHub token: a private sd-scripts fork, or a higher clone rate limit
#   HF_TOKEN         HuggingFace token: only needed for a gated/private base model
#   SD_SCRIPTS_REPO  optional: override the sd-scripts repo URL
# Nothing secret is written in this notebook.
from google.colab import userdata

def secret(name, default=''):
    try:
        return userdata.get(name) or default
    except Exception:
        return default  # the secret is simply not set

GH_TOKEN = secret('GH_TOKEN')
HF_TOKEN = secret('HF_TOKEN')
SD_SCRIPTS_REPO = secret('SD_SCRIPTS_REPO', 'https://github.com/kohya-ss/sd-scripts.git')

print('GH_TOKEN:', 'nakita' if GH_TOKEN else 'wala (anonymous clone)')
print('HF_TOKEN:', 'nakita' if HF_TOKEN else 'wala (public model lang)')
print('sd-scripts repo:', SD_SCRIPTS_REPO)


In [ ]:
# CELL 2: Mount Google Drive (the bridge from Server-2)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/LORA_TRAINING"   # Commander uploads C:\LORA_TRAINING here
assert os.path.isdir(DRIVE_ROOT), f"Missing folder: {DRIVE_ROOT}. Check the upload."
print("Dataset folders found:", sorted(os.listdir(f"{DRIVE_ROOT}/datasets")))

In [ ]:
# CELL 4: Install the trainer (kohya-ss sd-scripts) using GH_TOKEN
import os
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN   # so a gated HuggingFace model can download

%cd /content
clone_url = SD_SCRIPTS_REPO
if GH_TOKEN and 'github.com' in clone_url and '@' not in clone_url:
    clone_url = clone_url.replace('https://', 'https://' + GH_TOKEN + '@')
!git clone -q "$clone_url" /content/sd-scripts || git -C /content/sd-scripts pull -q
# Remove the token from the VM's git config again.
!git -C /content/sd-scripts remote set-url origin "$SD_SCRIPTS_REPO"
%cd /content/sd-scripts
!pip install -q -r requirements.txt
!pip install -q bitsandbytes


In [ ]:
# CELL 4: Training configuration (edit these only)
import os, glob

BASE_MODEL = f"{DRIVE_ROOT}/models/base_model.safetensors"  # base checkpoint on Drive
DATASETS   = f"{DRIVE_ROOT}/datasets"                        # one subfolder per LoRA
OUTPUT     = f"{DRIVE_ROOT}/output"                          # trained LoRAs go here
os.makedirs(OUTPUT, exist_ok=True)

TRAIN_ARGS = {
    "resolution": "512,512",
    "network_module": "networks.lora",
    "network_dim": 32,
    "network_alpha": 16,
    "learning_rate": "1e-4",
    "max_train_epochs": 10,
    "train_batch_size": 1,
    "mixed_precision": "fp16",        # T4 supports fp16, not bf16
    "optimizer_type": "AdamW8bit",
    "save_every_n_epochs": 2,
    "save_model_as": "safetensors",
    "cache_latents": True,
    "gradient_checkpointing": True,
    "sdpa": True,
}

In [ ]:
# CELL 5: Train every dataset, saving to Drive as it goes
import subprocess, os

def build_cmd(dataset_dir, name):
    cmd = ["accelerate", "launch", "--num_cpu_threads_per_process", "1",
           "train_network.py", "--pretrained_model_name_or_path", BASE_MODEL,
           "--train_data_dir", dataset_dir,
           "--output_dir", OUTPUT, "--output_name", name,
           "--logging_dir", f"{OUTPUT}/logs"]
    for k, v in TRAIN_ARGS.items():
        if isinstance(v, bool):
            if v: cmd.append(f"--{k}")
        else:
            cmd += [f"--{k}", str(v)]
    return cmd

for dataset_dir in sorted(glob.glob(f"{DATASETS}/*")):
    name = os.path.basename(dataset_dir)
    if os.path.exists(f"{OUTPUT}/{name}.safetensors"):
        print(f"SKIP (already trained): {name}")   # lets you resume after a disconnect
        continue
    print(f"TRAINING: {name}")
    result = subprocess.run(build_cmd(dataset_dir, name), capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FAIL: {name}\n", result.stderr[-1500:])
    else:
        print(f"PASS: {name}")

In [ ]:
# CELL 6: Zip final weights and save back to Drive for download
import zipfile, glob, os

ZIP_PATH = f"{DRIVE_ROOT}/LORA_ARSENAL_OUTPUT.zip"
files = glob.glob(f"{OUTPUT}/*.safetensors")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, arcname=os.path.basename(f))
print(f"Packaged {len(files)} LoRA files -> {ZIP_PATH}")
print("Commander: download this zip from Google Drive to Server-2.")